# Chapter 6 — Diagram customization

Engineer course · source candidate · CONVERGING

# Chapter 6 — Diagram customization

This complete Chapter is one clean-kernel execution unit. Its four web
Lessons are reading views over the ordered source fragments. The
required API changes presentation only; the circuit Plan remains
authoritative.

## Lesson 1 — Start from Default composition

### Declare one fixed four-arm Plan

Diagram composition changes presentation, never the circuit. This
illustration has one central Bus, four real capacitor arms, and four
terminated 50 ohm Ports.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    DiagramAxis,
    DiagramSide,
    RLGC,
    SCNSimValidationError,
    SchematicComposition,
    Theme,
    components,
    units as u,
)

four_arm_plan = CircuitPlan(id="four_arm_composition")
central_bus = four_arm_plan.bus(id="central")
outer_buses = {
    name: four_arm_plan.bus(id=name) for name in ("a", "b", "c", "d")
}
capacitors = {
    name: four_arm_plan.add(
        components.capacitor(id=f"capacitor_{name}", capacitance=value * u.fF)
    )
    for name, value in zip(("a", "b", "c", "d"), (4.0, 5.0, 6.0, 7.0), strict=True)
}
arms = {
    name: four_arm_plan.series(
        id=f"arm_{name}",
        start=central_bus,
        elements=(capacitors[name],),
        end=outer_buses[name],
    )
    for name in ("a", "b", "c", "d")
}
ports = {
    name: four_arm_plan.add_port(
        id=f"port_{name}",
        at=outer_buses[name],
        role="terminated",
        reference_impedance=50.0 * u.ohm,
    )
    for name in ("a", "b", "c", "d")
}

The branch labels are identifiers, not a claim about signal flow. The
central Bus adds no fifth arm.

In [ ]:
four_arm_composition = SchematicComposition.automatic(plan=four_arm_plan)
four_arm_default = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=four_arm_composition, theme=Theme.AUTO)
)
four_arm_default.show()

In [ ]:
four_arm_default.audit.show()

The A/B audit reconstructs visible electrical and structural evidence.
Passing it does not make pixels authoritative or reveal hidden recipe
metadata.

## Lesson 2 — Set complete Port poses

### Understand the eight valid combinations

A Port pose has a marker-facing `boundary_side` and a perpendicular
`load_side`. The valid pairs are left/top, left/bottom, right/top,
right/bottom, top/left, top/right, bottom/left, and bottom/right. The
complete setter updates both directions atomically; it does not change
the Port load.

In [ ]:
four_arm_composition.axis(arms["a"], DiagramAxis.HORIZONTAL)
four_arm_composition.axis(arms["b"], DiagramAxis.HORIZONTAL)
four_arm_composition.axis(arms["c"], DiagramAxis.VERTICAL)
four_arm_composition.axis(arms["d"], DiagramAxis.VERTICAL)
four_arm_composition.port_orientation(
    ports["a"], boundary_side=DiagramSide.LEFT, load_side=DiagramSide.BOTTOM
)
four_arm_composition.port_orientation(
    ports["b"], boundary_side=DiagramSide.RIGHT, load_side=DiagramSide.BOTTOM
)
four_arm_composition.port_orientation(
    ports["c"], boundary_side=DiagramSide.TOP, load_side=DiagramSide.RIGHT
)
four_arm_composition.port_orientation(
    ports["d"], boundary_side=DiagramSide.BOTTOM, load_side=DiagramSide.RIGHT
)
four_arm_composition.show()

This lesson uses four of the eight valid poses once. It does not search
all poses or reinterpret a Port’s boundary marker as its circuit
attachment.

## Lesson 3 — Request actual T and Cross junctions

### Build a three-contact tapped feedline

The T example reuses the established two-section scalar CPW declaration.
Its three contacts are the left-section end, right-section start, and
exposed tap Pin on one owner-local Bus.

In [ ]:
tapped_plan = CircuitPlan(id="tapped_feedline_t")
feedline = tapped_plan.subsystem(id="feedline")
cpw = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)
left = feedline.add(
    components.transmission_line(
        id="left", length=1.0 * u.mm, rlgc=cpw, n_sections=1
    )
)
right = feedline.add(
    components.transmission_line(
        id="right", length=1.0 * u.mm, rlgc=cpw, n_sections=1
    )
)
input_bus = feedline.bus(id="input")
tap_bus = feedline.bus(id="tap")
output_bus = feedline.bus(id="output")
left_tap = tap_bus.tap(id="left_section")
right_tap = tap_bus.tap(id="right_section")
coupling_tap = tap_bus.tap(id="coupling")
left_section = feedline.series(
    id="left_section",
    start=input_bus,
    elements=(left.between(left.pin("head", conductor="signal"), left.pin("tail", conductor="signal")),),
    end=left_tap,
)
right_section = feedline.series(
    id="right_section",
    start=right_tap,
    elements=(right.between(right.pin("head", conductor="signal"), right.pin("tail", conductor="signal")),),
    end=output_bus,
)
feedline.expose_pin(id="input", at=input_bus)
feedline_tap_pin = feedline.expose_pin(id="tap", at=coupling_tap)
feedline.expose_pin(id="output", at=output_bus)

In [ ]:
tapped_composition = SchematicComposition.automatic(plan=tapped_plan)
tap_wiring = tapped_composition.replace_wiring(at=tap_bus)
tap_t = tap_wiring.tee(id="tap_t", branch=DiagramSide.BOTTOM)
tap_wiring.connect(
    tapped_composition.endpoint(left_section, boundary="end"), tap_t.left
)
tap_wiring.connect(
    tapped_composition.endpoint(right_section, boundary="start"), tap_t.right
)
tap_wiring.connect(tapped_composition.endpoint(feedline_tap_pin), tap_t.bottom)
tapped_t_diagram = None
try:
    tapped_t_diagram = tapped_plan.render_schematic(
        CircuitDiagramSpec(layout=tapped_composition, theme=Theme.AUTO)
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    display({"T layout unavailable": str(error), "stage": error.stage})
if tapped_t_diagram is not None:
    display(tapped_t_diagram.show())

The T is a non-element conductive presentation primitive. It creates no
new net, Tap, Coordinate, or permission to address private Composite
contents.

### Replace the four-contact group with one Cross

In [ ]:
central_wiring = four_arm_composition.replace_wiring(at=central_bus)
central_cross = central_wiring.cross(id="central_cross")
central_wiring.connect(
    four_arm_composition.endpoint(arms["a"], boundary="start"), central_cross.left
)
central_wiring.connect(
    four_arm_composition.endpoint(arms["b"], boundary="start"), central_cross.right
)
central_wiring.connect(
    four_arm_composition.endpoint(arms["c"], boundary="start"), central_cross.top
)
central_wiring.connect(
    four_arm_composition.endpoint(arms["d"], boundary="start"), central_cross.bottom
)
four_arm_cross = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=four_arm_composition, theme=Theme.AUTO)
)
four_arm_cross.show()

In [ ]:
four_arm_cross.audit.show()

A pair of Ts might preserve electrical connectivity but would not
satisfy this requested Cross shape. Every attachment and requested arm
is used once.

## Lesson 4 — Reuse the completed recipe

### Convert captured choices back to an editable composition

The captured composition is detached and immutable. `to_composition()`
checks the exact Plan identity and returns a new editable recipe; it
does not freeze pixels.

In [ ]:
fixed_four_arm = four_arm_cross.composition.to_composition(plan=four_arm_plan)
fixed_four_arm.show()
four_arm_fixed = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=fixed_four_arm, theme=Theme.AUTO)
)
four_arm_fixed.show()

In [ ]:
four_arm_fixed.audit.show()

Rendering remeasures current labels and geometry. An incompatible
parameter point can therefore fail rather than silently search for
another layout.